> 在线讲义：[第 17 章 · 生成模型的数学：ELBO、两种梯度估计、连续性方程与 flow matching 目标](https://wikieden.github.io/math-and-eval-stats/lessons/17-generative-math.html) · 本书《数学与实验统计》

# 第 17 章 · 生成模型的数学：ELBO、两种梯度估计、连续性方程与 flow matching 目标

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/17-generative-math.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 ELBO | 闭式 | `log_evidence()`、`elbo(m, s)`、`kl_gauss(...)` | −1.6065；差值 = KL |
| 第二步 梯度估计 | 重参数化 / score | `grad_estimators()` | 标准差 2.00 / 5.51 / 4.28 |
| 第三步 连续性方程 | 有限差分 | `continuity_residual(0.5, 0.3)` | 相加 −7.9e-8 |
| 第四步 FM vs CFM | 两个目标 | `fm_vs_cfm()` | 差 0.812–0.814 |

例子：一维潜变量模型 z ~ N(0, 1)、x | z ~ N(z, 0.5²)，观测 x = 1.2；∇_μ E[z²] 的两种梯度估计；
噪声 N(0, 1) 到数据 N(2, 0.5²) 的直线插值，验证连续性方程和"条件目标与边际目标只差常数"。
里程碑 ME-M13 的一维两峰 score 和 flow matching 训练留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
import numpy as np
SIG_X = 0.5   # 似然的标准差
X_OBS = 1.2
M1, S1 = 2.0, 0.5  # 数据分布 N(2, 0.5²)

**函数 `log_normal`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def log_normal(x, m, s):
    return -0.5 * math.log(2 * math.pi * s * s) - 0.5 * ((x - m) / s) ** 2

**函数 `log_evidence`**：log p(x)：z 积分掉以后 x ~ N(0, 1 + σ²)。

In [ ]:
def log_evidence(x: float = X_OBS) -> float:
    """log p(x)：z 积分掉以后 x ~ N(0, 1 + σ²)。"""
    return log_normal(x, 0.0, math.sqrt(1 + SIG_X**2))

**函数 `posterior`**：真后验 p(z | x) = N(x/(1 + σ²), σ²/(1 + σ²))，返回 (均值, 标准差)。

In [ ]:
def posterior(x: float = X_OBS) -> tuple[float, float]:
    """真后验 p(z | x) = N(x/(1 + σ²), σ²/(1 + σ²))，返回 (均值, 标准差)。"""
    v = SIG_X**2 / (1 + SIG_X**2)
    return x / (1 + SIG_X**2), math.sqrt(v)

**函数 `elbo`**：q = N(m, s²) 时的 ELBO = E_q[log p(x|z)] − KL(q ‖ N(0, 1))，两项都有闭式。

In [ ]:
def elbo(m: float, s: float, x: float = X_OBS) -> float:
    """q = N(m, s²) 时的 ELBO = E_q[log p(x|z)] − KL(q ‖ N(0, 1))，两项都有闭式。"""
    rec = -0.5 * math.log(2 * math.pi * SIG_X**2) - ((x - m) ** 2 + s * s) / (2 * SIG_X**2)
    kl = 0.5 * (m * m + s * s - 1 - math.log(s * s))
    return rec - kl

**函数 `kl_gauss`**：KL(N(m1, s1²) ‖ N(m2, s2²))。

In [ ]:
def kl_gauss(m1, s1, m2, s2) -> float:
    """KL(N(m1, s1²) ‖ N(m2, s2²))。"""
    return math.log(s2 / s1) + (s1 * s1 + (m1 - m2) ** 2) / (2 * s2 * s2) - 0.5

**函数 `grad_estimators`**：估 ∇_μ E_{z~N(μ,1)}[z²]（真值 2μ）。返回 {名字: (均值, 单样本标准差)}。

In [ ]:
def grad_estimators(mu: float = 1.0, n: int = 100000, seed: int = 0) -> dict[str, tuple[float, float]]:
    """估 ∇_μ E_{z~N(μ,1)}[z²]（真值 2μ）。返回 {名字: (均值, 单样本标准差)}。"""
    eps = np.random.default_rng(seed).standard_normal(n)
    z = mu + eps
    reparam = 2 * z                       # d/dμ (μ + ε)² = 2(μ + ε)
    score = z**2 * (z - mu)               # f(z) · ∇_μ log N(z; μ, 1)
    base = (z**2 - (mu**2 + 1)) * (z - mu)  # 减去基线 E[z²] = μ² + 1
    return {k: (float(v.mean()), float(v.std())) for k, v in (("重参数化", reparam), ("score function", score), ("score function + 基线", base))}

**函数 `path_stats`**：x_t = (1 − t)·x₀ + t·x₁，x₀ ~ N(0, 1)、x₁ ~ N(M1, S1²) 独立：x_t ~ N(t·M1, (1 − t)² + t²·S1²)。返回 (均值, 标准差)。

In [ ]:
def path_stats(t: float) -> tuple[float, float]:
    """x_t = (1 − t)·x₀ + t·x₁，x₀ ~ N(0, 1)、x₁ ~ N(M1, S1²) 独立：x_t ~ N(t·M1, (1 − t)² + t²·S1²)。返回 (均值, 标准差)。"""
    return t * M1, math.sqrt((1 - t) ** 2 + t * t * S1 * S1)

**函数 `marginal_velocity`**：高斯路径的边际速度 v = μ'_t + (σ'_t/σ_t)·(x − μ_t)。

In [ ]:
def marginal_velocity(x, t: float):
    """高斯路径的边际速度 v = μ'_t + (σ'_t/σ_t)·(x − μ_t)。"""
    mu, sd = path_stats(t)
    dsd = (-(1 - t) + t * S1 * S1) / sd
    return M1 + dsd / sd * (x - mu)

**函数 `continuity_residual`**：有限差分核对连续性方程 ∂p/∂t + ∂(p·v)/∂x = 0。返回 (∂p/∂t, ∂(pv)/∂x)。

In [ ]:
def continuity_residual(x: float, t: float, h: float = 1e-4) -> tuple[float, float]:
    """有限差分核对连续性方程 ∂p/∂t + ∂(p·v)/∂x = 0。返回 (∂p/∂t, ∂(pv)/∂x)。"""
    p = lambda xx, tt: math.exp(log_normal(xx, *path_stats(tt)))
    dpdt = (p(x, t + h) - p(x, t - h)) / (2 * h)
    flux = lambda xx: p(xx, t) * marginal_velocity(xx, t)
    return dpdt, (flux(x + h) - flux(x - h)) / (2 * h)

**函数 `fm_vs_cfm`**：对一族 v_c(x) = c·x：条件目标 E‖v_c − (x₁ − x₀)‖² 和边际目标 E‖v_c − v(x)‖²，差值和 c 无关。返回 [(c, CFM, FM, 差)]。

In [ ]:
def fm_vs_cfm(t: float = 0.5, cs=(0.0, 0.5, 1.0, 2.0), n: int = 400000, seed: int = 0):
    """对一族 v_c(x) = c·x：条件目标 E‖v_c − (x₁ − x₀)‖² 和边际目标 E‖v_c − v(x)‖²，差值和 c 无关。返回 [(c, CFM, FM, 差)]。"""
    rng = np.random.default_rng(seed)
    x0, x1 = rng.standard_normal(n), rng.normal(M1, S1, n)
    xt = (1 - t) * x0 + t * x1
    out = []
    for c in cs:
        cfm = float(((c * xt - (x1 - x0)) ** 2).mean())
        fm = float(((c * xt - marginal_velocity(xt, t)) ** 2).mean())
        out.append((c, cfm, fm, cfm - fm))
    return out

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 ELBO：log p(x) = ELBO + KL(q ‖ 真后验)")
pm, ps = posterior()
print(f"  log p(x = 1.2) = {log_evidence():.4f}；真后验 N({pm:.2f}, {ps:.4f}²)")
for m, s in ((pm, ps), (0.0, 1.0), (1.5, 0.3)):
    e = elbo(m, s)
    print(f"  q = N({m:.2f}, {s:.2f}²)：ELBO = {e:.4f}，差 {log_evidence() - e:.4f}，KL(q‖后验) = {kl_gauss(m, s, pm, ps):.4f}")

In [ ]:
print("\n第二步 两种梯度估计：∇_μ E[z²]，μ = 1，真值 2")
for k, (mean, sd) in grad_estimators().items():
    print(f"  {k}：均值 {mean:.3f}，单样本标准差 {sd:.2f}")

In [ ]:
print("\n第三步 连续性方程：速度场推着密度走")
for x, t in ((0.5, 0.3), (1.5, 0.7)):
    a, b = continuity_residual(x, t)
    print(f"  x = {x}、t = {t}：∂p/∂t = {a:+.5f}，∂(pv)/∂x = {b:+.5f}，相加 {a + b:+.1e}")

In [ ]:
print("\n第四步 flow matching：条件目标和边际目标只差一个常数")
for c, cfm, fm, d in fm_vs_cfm():
    print(f"  v = {c}·x：条件目标 {cfm:.4f}，边际目标 {fm:.4f}，差 {d:.4f}")

## 3. 代码里才看得到的

1. **E_q[(x − z)²] = (x − m)² + s²**：重建项的期望里多出的 s² 是 q 的方差，这一项让 ELBO 不喜欢太宽的 q。
2. **连续性方程用中心差分**：步长 h = 10⁻⁴，误差 O(h²) ≈ 10⁻⁸，所以残差在 10⁻⁷ 量级就算通过。
3. **FM 和 CFM 用同一批样本比**：两种目标在同一批 (x₀, x₁) 上算，蒙特卡洛误差大部分抵消，差值才能稳在千分位。

## 4. 自己改一改

1. 把 `SIG_X` 改成 0.1（观测很准），真后验变成什么样？先验 q = N(0, 1) 的 ELBO 差多少？
2. 在 `grad_estimators` 里把 f 换成 z⁴，重参数化和 score function 的方差各变成多少？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> log p 等于 ELBO 加 KL，推高 ELBO 一举两得；
> 重参数化把随机挪出去，score function 只看概率，方差大要基线；
> 墨水跟着水流走，浓度变化等于净流入；
> 每辆车猜自己的方向，学到的就是路口的平均。